In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

## 📍 Lodaing **`all-MiniLM-L6-v2`** Model

In [ ]:
model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
# model.save("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 📍 Loading Datasets

In [12]:
jobs = pd.read_csv(r"D:\My Download\Elevvo pathway\Elevvo pathway Tasks\Dataset\Jobs.csv")
companies = pd.read_csv(r"D:\My Download\Elevvo pathway\Elevvo pathway Tasks\Dataset\Companies.csv")

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Computec\AppData\Local\Temp\ipykernel_25756\582971194.py:1: SyntaxWarning: invalid escape sequence '\M'
  jobs = pd.read_csv("D:\My Download\Elevvo pathway\Elevvo pathway Tasks\Dataset\Jobs.csv")
C:\Users\Computec\AppData\Local\Temp\ipykernel_25756\582971194.py:2: SyntaxWarning: invalid escape sequence '\M'
  companies = pd.read_csv("D:\My Download\Elevvo pathway\Elevvo pathway Tasks\Dataset\Companies.csv")


In [13]:
np.unique(jobs["Qualifications"])

array(['B.Com', 'B.Tech', 'BA', 'BBA', 'BCA', 'M.Com', 'M.Tech', 'MBA',
       'MCA', 'PhD'], dtype=object)

## 🟢 filtering job descriptions to reduce the number of irrelevant ones

In [14]:
filterd_data = jobs[(jobs["Job Title"] == "Teacher") & (jobs["Work Type"] == "Intern") & ((jobs["Preference"] == "Male") | (jobs["Preference"] == "Both")) ]
filterd_companies = companies[companies["Job Id"].isin(filterd_data["Job Id"].to_list())]

## 🟢 Employer cv Description 

In [15]:
employer_description = 'master teacher highlights home schooling knowledge calm and patient certified in early childhood education head start programs strong communicator toddler and preschool curricula classroom management classroom management skills i have a lot patience  i communicate really well with children  i can usually calm a stressful situation quickly  education early childhood education      san jacinto college     city   state   usa coursework in child nutritionchild abuse awareness trainingcoursework in emergency preparednesscoursework in behavior managementemphasis in child development high school diploma   general      beaumont high school     city   state   usa emphasis in child development experience master teacher sep      to apr       company name     city   state conducted small group and individual classroom activities based on differentiated learning needs encouraged students to be understanding of and helpful to others observed students to supply teachers with feedback regarding potential learning blocks and opportunities for support physically and verbally interacted with students throughout the day to keep them engaged supported students in developing strategies for individual needs and classroom group dynamics organized field trips to local parks  fire stations and zoos helped prepare daily lesson plans for activities and lessons implemented emergent curriculum to encourage student participation supplied one on one attention to each student  while maintaining overall focus on the entire group wrote daily and weekly lesson plans administered minor first aid to injured students applied the positive reinforcement method to redirect negative behaviors promoted language development skills through reading and storytelling  implemented family style meals  conducting parent teacher conferences  and kept personal profiles of each child  teachers associate aug      to nov       company name     city   state promoted language development skills through reading and storytelling applied the positive reinforcement method to redirect negative behaviors conducted small group and individual classroom activities based on differentiated learning needs assisted     children per station during small group learning periods  organized field trips to encouraged students to be understanding of and helpful to others observed students to supply teachers with feedback regarding potential learning blocks and opportunities for support physically and verbally interacted with students throughout the day to keep them engaged helped prepare daily lesson plans for activities and lessons implemented emergent curriculum supplied one on one attention to each student  while maintaining overall focus on the entire group physically and verbally interacted with students throughout the day to keep them engaged communicated effectively with educators from various grade levels wrote daily and weekly lesson plans administered minor first aid to injured students to encourage student participation organized field trips to local parks  fire stations and zoos maintained daily records of children s individual activities  behaviors  meals and naps  established a safe play environment for the children distributed quarterly educational assessments  similar to report cards  to each parent developed professional relationships with parents  teachers  directors and therapists observed children for signs of illness  injury  emotional disturbance  learning disorders and speech problems maintained daily records of children s individual activities  behaviors  meals and naps  teacher aide aug      to mar       company name     city   state promoted language development skills through reading and storytelling applied the positive reinforcement method to redirect negative behaviors conducted small group and individual classroom activities based on differentiated learning needs  organized field trips to local parks  fire stations and zoos  assisted     children per station during small group learning periods encouraged students to be understanding of and helpful to others observed students to supply teachers with feedback regarding potential learning blocks and opportunities for support physically and verbally interacted with students throughout the day to keep them engaged supported students in developing strategies for individual needs and classroom group dynamics helped prepare daily lesson plans for activities and lessons  implemented emergent curriculum to encourage student participation  supplied one on one attention to each student  while maintaining overall focus on the entire group communicated effectively with educators from various grade levels wrote daily and weekly lesson plans administered minor first aid to injured students maintained daily records of children s individual activities  behaviors  meals and naps established a safe play environment for the children created and implemented developmentally appropriate curriculum that addressed all learning styles distributed quarterly educational assessments  similar to report cards  to each parent maintained a child friendly environment with access to outdoor activities completed all required documentation for the national head start program  extra curricular activities  i ran the nursery at my church for a yea and then taught sunday school for the older kids for another two years   i also helped run an after school  get your homework done here  program the community started where kids that didn t have help at home could come to us and get help with their homework or for the younger children the parents could enroll them and they would come daily after school  summary talented early education professional with diverse experience in planning and implementing various activities for promoting physical  social  emotional and intellectual growth of children  '

## ✔ Embedding Jobs Descriptions & Employer Description

In [16]:
jd_embedding = model.encode([employer_description])
resume_embeddings = model.encode(filterd_data["Job Description"].to_list())

## ✔ Calculate Cosine Similarity
* This returns a score between **0 and 1** representing semantic closeness

In [17]:
similarity_scores = cosine_similarity(jd_embedding, resume_embeddings)[0]

## 🚩 Rank and Format the Results

In [18]:
results = []
for i, score in enumerate(similarity_scores):
    job_data = filterd_data.iloc[i]
    company_data = filterd_companies.iloc[i]

    results.append({
        "Job Description": job_data["Job Description"],
        "Salary Range":job_data['Salary Range'],
        "Benefits":job_data['Benefits'],
        "skills":job_data['skills'],
        "Contact Person":company_data['Contact Person'],
        "Contact":company_data['Contact'],
        "Responsibilities":job_data['Responsibilities'],

        "Company" :company_data["Company"],
        "Company Profile" :company_data["Company Profile"],
        "Company Size" :company_data["Company Size"],
        "location" :company_data["location"],
        "Country" :company_data["Country"],
        "latitude" :company_data["latitude"],
        "longitude" :company_data["longitude"],
        "Date":job_data['Date'],
        "Preference": job_data["Preference"],
        "Match_Score": round(score * 100, 2), 
    })


## 🚩 Convert to Pandas DataFrame for a clean, analytical output

In [ ]:
pd.DataFrame(results).sort_values(by="Match_Score", ascending=False)

## 📍 Functions

In [20]:
def SimilarityCVwithJobs(Jobs:pd.DataFrame,Companies:pd.DataFrame,Employer_Desc:str):
    model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
    filterd_Jobs = Jobs[(Jobs["Job Title"] == "Teacher") & (Jobs["Work Type"] == "Intern") & (Jobs["Preference"] == "Male" or Jobs["Preference"] == "Both" ) ]
    filterd_companies = Companies[Companies["Job Id"].isin(filterd_Jobs["Job Id"].to_list())]

    jd_embedding = model.encode([Employer_Desc])
    resume_embeddings = model.encode(filterd_Jobs["Job Description"].to_list())
    
    return cosine_similarity(jd_embedding, resume_embeddings)[0] , filterd_Jobs , filterd_companies

In [21]:
def GetTopJobs(Similarity_Scores:np.ndarray,Filterd_Jobs:pd.DataFrame,Filterd_Companies:pd.DataFrame):
    results = []
    for i, score in enumerate(Similarity_Scores):
        job_data = Filterd_Jobs.iloc[i]
        company_data = Filterd_Companies.iloc[i]

        results.append({
            "Job Description": job_data["Job Description"],
            "Salary Range":job_data['Salary Range'],
            "Benefits":job_data['Benefits'],
            "skills":job_data['skills'],
            "Contact Person":company_data['Contact Person'],
            "Contact":company_data['Contact'],
            "Responsibilities":job_data['Responsibilities'],

            "Company" :company_data["Company"],
            "Company Profile" :company_data["Company Profile"],
            "Company Size" :company_data["Company Size"],
            "location" :company_data["location"],
            "Country" :company_data["Country"],
            "latitude" :company_data["latitude"],
            "longitude" :company_data["longitude"],
            "Date":job_data['Date'],
            "Match_Score": round(score * 100, 2), 
        })
    
    return pd.DataFrame(results).sort_values(by="Match_Score", ascending=False)


## Skills Column Preprocessing

In [82]:
skills = jobs[jobs["Job Title"] == "Data Scientist"]["skills"]
skills.iloc[:100]


328      Machine learning algorithms Python programming...
701      Data integration ETL (Extract, Transform, Load...
1193     Machine learning algorithms Python programming...
1262     Data integration ETL (Extract, Transform, Load...
1448     Data integration ETL (Extract, Transform, Load...
                               ...                        
26000    Machine learning algorithms Python programming...
26108    Data integration ETL (Extract, Transform, Load...
26197    Machine learning algorithms Python programming...
26361    Machine learning algorithms Python programming...
26628    Data integration ETL (Extract, Transform, Load...
Name: skills, Length: 100, dtype: object

In [87]:
skills.iloc[1:].apply(lambda x: re.findall(r"\b[A-Z]+[a-z]*(?:\s+[a-z]+)*",string=x)).to_list()[0]

['Data integration',
 'ETL',
 'Extract',
 'Transform',
 'Load',
 'Big data technologies',
 'Hadoop',
 'Spark',
 'Database management',
 'Data warehousing']

---

<h1 align="center">✅ The End of Model Notebook</h1>


---